# `wf_external_transfer` 단계별 Testbed

이 Notebook은 타인송금 Workflow를 한 번에 실행하고 결과만 보는 용도가 아니다. 각 단계에서 다음 내용을 차례대로 확인한다.

1. Agent가 받은 사용자 발화
2. Agent 내부 수취인 이름 힌트·금액 Slot 추출 결과
3. Agent가 Backend Tool API에 보낸 요청과 Backend 응답 (수취인 확정 → 출금 계좌 확인)
4. Agent가 Backend로 보낸 수취인·계좌 선택 UI Webhook
5. Backend가 검증하여 Agent에 전달한 Resume 값
6. Prepare → 승인 → 인증 → 실행까지 이어지는 나머지 Step

본인이체와 다른 점은 맨 앞에 **수취인을 확정하는 절차**가 붙는다는 것뿐이다. 기본 Scenario는 **수취인 후보가 여럿이라 사용자가 새 후보를 선택하고, 출금 계좌도 선택이 필요한 경우**다 — 신규 후보 수취인(`to_recipient_candidate_id`)과 기존 수취인(`to_recipient_id`) 두 참조 방식 중 전자를 보여준다. Cell을 위에서부터 하나씩 실행한다.

## 전체 실행 흐름

```text
사용자 발화 ("철수에게 10만원 보내줘")
  → [Agent 내부] 수취인 이름 힌트·금액 추출 ("철수" 발견)
  → [Agent → Backend API] 수취인 확정 → selection_required
  → [Agent → Backend Webhook] 수취인 선택 UI 요청
  → Workflow 중단 (1차)
  → [Backend → Agent] 신규 후보(to_recipient_candidate_id)로 Resume
  → [Agent → Backend API] 출금 계좌 확인 → selection_required
  → [Agent → Backend Webhook] 출금 계좌 선택 UI 요청
  → Workflow 중단 (2차)
  → [Backend → Agent] 검증된 from_account_id로 Resume
  → [Agent → Backend API] 송금 Prepare → ready_for_confirmation
  → [Agent → Backend Webhook] 승인 요청
  → Workflow 중단 (3차)
  → [Backend → Agent] 승인(approved)으로 Resume
  → [Agent → Backend API] 인증 Context 생성 → authentication_required
  → [Agent → Backend Webhook] 인증 요청
  → Workflow 중단 (4차)
  → [Backend → Agent] 인증 성공(verified)으로 Resume
  → [Agent → Backend API] 송금 실행 → completed
  → [Agent → Backend Webhook] 결과 전송
```

수취인 이름 힌트가 발화에 없으면 `resolve_recipient_hint` 호출 자체를 건너뛰고 바로 선택 화면으로 간다 — 이 분기는 `agent/tests/test_external_transfer_reference_workflow.py`의 `test_external_transfer_no_recipient_hint_skips_resolve_call`에서 별도로 검증한다.

## 0. 공통 설정과 출력 함수

In [ ]:
import json
import os

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing import MockBackend
from agent.testing.external_transfer import (
    create_external_transfer_backend_testbed,
    create_external_transfer_mock_testbed,
)
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.external_transfer import extract_external_transfer_slots_from_text


def show_json(title, value):
    print(chr(10) + f"--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


CHAT_SESSION_ID = "chat_testbed_003"
EXECUTION_CONTEXT_ID = "exec_testbed_003"
THREAD_ID = "thread_external_testbed_001"

print("공통 설정 완료")

## 1. 관리시트 계약에서 실행 Step 확인

Notebook에서 임의로 Workflow 순서를 정하지 않는다. 관리시트에서 생성한 Manifest를 읽어 현재 Step, 통신 방식과 계약 ID를 확인한다.

In [ ]:
store = WorkflowContractStore()
workflow_contract = store.get_workflow("wf_external_transfer")

step_contracts = [
    {
        "order": step["step_order"],
        "step_id": step["step_id"],
        "mode": step["interaction_mode"],
        "contract_id": step.get("contract_id"),
        "next": step["route_summary"],
    }
    for step in workflow_contract["steps"]
]
show_json("Workflow Step 계약", step_contracts)

## 2. Step 1 — 사용자 발화에서 Slot 추출

### 입력

사용자가 수취인 이름과 금액을 함께 말한다. 이 Step은 Agent 내부에서만 실행되며 API 호출은 없다. 출금 계좌 힌트는 여기서 추출하지 않는다 — 수취인이 확정된 다음에야 의미가 있기 때문이다.

기준 입력은 `철수에게 10만원 보내줘`다.

In [ ]:
USER_MESSAGE = "철수에게 10만원 보내줘"

slot_input = {"message": USER_MESSAGE}
slot_output = dict(extract_external_transfer_slots_from_text(USER_MESSAGE))

show_json("Step 1 입력", slot_input)
show_json("Step 1 출력", slot_output)

assert slot_output == {"recipient_name_hint": "철수", "amount": 100000}

### 다른 발화를 직접 넣어보기

이 Cell은 Workflow 실행에 영향을 주지 않는 연습용이다. 이름 힌트가 없으면 `recipient_name_hint`가 `None`이 되고, 이 경우 Workflow는 `resolve_recipient_hint` 호출 없이 바로 선택 화면으로 간다.

In [ ]:
practice_messages = [
    "영희한테 5만원 송금해줘",
    "10만원 송금해줘",
    "송금해줘",
]

for message in practice_messages:
    show_json(message, dict(extract_external_transfer_slots_from_text(message)))

## 3. Mock Backend 입력값 준비

Backend가 아직 없어도 실제 API와 같은 응답 Schema로 Workflow를 실행한다. 여기서는 수취인 확정이 `selection_required`(이름 후보 여럿)를 반환하고, 사용자가 신규 후보를 선택하는 경로를 사용한다. 이어서 출금 계좌도 후보가 둘이라 선택이 필요하다.

아래 값은 Agent가 만든 값이 아니라 **Backend가 검증한 뒤 반환했다고 가정한 값**이다.

In [ ]:
def account(account_id, alias, bank_name="신한은행"):
    return {
        "account_id": account_id,
        "bank_name": bank_name,
        "account_alias": alias,
        "account_type": "checking",
        "masked_account_number": "110-***-123456",
        "currency": "KRW",
        "is_default": False,
        "status": "active",
    }


FROM_ACCOUNT_CANDIDATES = [
    account("acc_001", "생활비 통장"),
    account("acc_002", "비상금 통장"),
]
SELECTED_FROM_ACCOUNT_ID = "acc_001"
SELECTED_RECIPIENT_CANDIDATE_ID = "candidate_555"

CONFIRMATION_VIEW = {
    "from_account": {
        "account_id": SELECTED_FROM_ACCOUNT_ID,
        "bank_name": "신한은행",
        "account_alias": "생활비 통장",
        "masked_account_number": "110-***-123456",
    },
    "recipient": {
        "name": "김철수",
        "bank_name": "국민은행",
        "masked_account_number": "222-***-456789",
    },
    "amount": 100000,
    "fee": 500,
    "total_debit": 100500,
    "currency": "KRW",
    "expires_at": "2026-07-17T10:05:00+09:00",
}

AUTH_REQUEST_VIEW = {
    "title": "본인 인증이 필요합니다.",
    "available_methods": ["biometric"],
    "expires_at": "2026-07-17T10:10:00+09:00",
}

show_json("출금 계좌 후보", FROM_ACCOUNT_CANDIDATES)
show_json("승인 화면 확인 내역", CONFIRMATION_VIEW)

## 4. Mock Backend와 실제 Workflow 연결

예상하지 않은 Method나 Path가 호출되면 Mock Backend가 즉시 실패한다. 따라서 등록한 응답과 실제 Workflow 호출이 일치하는지도 함께 검증된다.

In [ ]:
mock_backend = MockBackend()
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/recipients:resolve",
    {"outcome": "selection_required", "selection_reason": "multiple_matches"},
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_recipient_001"}
)
mock_backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    {
        "account_resolution_outcome": "selection_required",
        "accounts": FROM_ACCOUNT_CANDIDATES,
        "account_ids": [],
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_from_001"}
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/transfers/external:prepare",
    {
        "outcome": "ready_for_confirmation",
        "confirmation_id": "confirm_testbed_002",
        "confirmation_view": CONFIRMATION_VIEW,
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_approval_001"}
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/auth-contexts",
    {
        "outcome": "authentication_required",
        "auth_context_id": "auth_testbed_002",
        "auth_request_view": AUTH_REQUEST_VIEW,
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_auth_001"}
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/transfers/external",
    {
        "outcome": "completed",
        "transaction_id": "txn_testbed_002",
        "completed_at": "2026-07-17T10:11:00+09:00",
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_result_001"}
)

mock_config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("agent-service-token"),
    agent_webhook_secret=SecretStr("agent-webhook-secret"),
    retry_backoff_seconds=0,
)

testbed = create_external_transfer_mock_testbed(
    mock_backend, mock_config, thread_id=THREAD_ID
)
print("Mock Testbed 연결 완료")

## 5. Step 2·3 실행 — 수취인 확정 후 1차 중단

`start()`를 호출하면 다음 중단 지점까지 실행한다. 이름 힌트가 있으므로 `resolve_recipient_hint`를 호출하고, 후보가 여럿이라 선택 UI에서 멈춘다.

In [ ]:
start_request = {
    "message": USER_MESSAGE,
    "request_id": "req_external_start_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
}
show_json("Workflow 시작 입력", start_request)

waiting_recipient = await testbed.start(**start_request)
show_json("1차 중단 결과", waiting_recipient.model_dump(mode="json"))

assert waiting_recipient.status == "waiting"
assert waiting_recipient.pending_interaction["type"] == "input"
RECIPIENT_INPUT_REQUEST_ID = waiting_recipient.pending_interaction["input_request_id"]

### 실제 수취인 확정 API 요청과 선택 UI Webhook

In [ ]:
recipient_resolve_exchange = mock_backend.exchange_timeline(include_payload=True)[0]
show_json("Agent → Backend 수취인 확정 요청/응답", recipient_resolve_exchange)

recipient_input_webhook = mock_backend.exchange_timeline(include_payload=True)[1][
    "request"
]
show_json("Agent → Backend 수취인 선택 Webhook", recipient_input_webhook)

assert recipient_resolve_exchange["request"]["recipient_name_hint"] == "철수"
assert recipient_input_webhook["metadata"]["step_id"] == "request_recipient_selection"
assert (
    recipient_input_webhook["metadata"]["ui"]["payload"]["recipient_selection_reason"]
    == "multiple_matches"
)

## 6. Step 4·5 실행 — 출금 계좌 확인 후 2차 중단

Backend가 검증한 신규 후보 수취인(`to_recipient_candidate_id`)으로 Resume하면, Agent는 곧바로 출금 계좌 확인 API를 호출한다.

In [ ]:
resume_recipient_request = {
    "agent_thread_id": THREAD_ID,
    "request_id": "req_external_resume_recipient_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "input_request_id": RECIPIENT_INPUT_REQUEST_ID,
    "value": {
        "recipient_selection_outcome": "selected",
        "to_recipient_id": None,
        "to_recipient_candidate_id": SELECTED_RECIPIENT_CANDIDATE_ID,
    },
}
show_json("Backend → Agent 수취인 Resume", resume_recipient_request)

waiting_from = await testbed.resume_input(**resume_recipient_request)
show_json("2차 중단 결과", waiting_from.model_dump(mode="json"))

assert waiting_from.status == "waiting"
FROM_INPUT_REQUEST_ID = waiting_from.pending_interaction["input_request_id"]

### 실제 출금 계좌 확인 API 요청과 선택 UI Webhook

In [ ]:
from_account_exchange = mock_backend.exchange_timeline(include_payload=True)[2]
show_json("Agent → Backend 출금 계좌 확인 요청/응답", from_account_exchange)

from_input_webhook = mock_backend.exchange_timeline(include_payload=True)[3]["request"]
show_json("Agent → Backend 출금 계좌 선택 Webhook", from_input_webhook)

assert (
    from_input_webhook["metadata"]["step_id"]
    == "request_external_from_account_selection"
)

## 7. Step 6·7 실행 — Prepare 후 승인 대기(3차 중단)

수취인·출금 계좌가 모두 확정되면 Agent는 송금 Prepare API를 호출한다. 이때 요청에는 `to_recipient_candidate_id`만 들어가고 `to_recipient_id`는 담기지 않는다 — Backend 계약(`ExternalTransferPrepareRequest`)이 둘 중 정확히 하나만 허용하기 때문이다.

In [ ]:
resume_from_request = {
    "agent_thread_id": THREAD_ID,
    "request_id": "req_external_resume_from_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "input_request_id": FROM_INPUT_REQUEST_ID,
    "value": {
        "account_selection_outcome": "selected",
        "account_ids": [SELECTED_FROM_ACCOUNT_ID],
    },
}
show_json("Backend → Agent 출금 계좌 Resume", resume_from_request)

waiting_approval = await testbed.resume_input(**resume_from_request)
show_json("3차 중단 결과(승인 대기)", waiting_approval.model_dump(mode="json"))

assert waiting_approval.status == "waiting"
assert waiting_approval.pending_interaction["type"] == "approval"
CONFIRMATION_ID = waiting_approval.pending_interaction["confirmation_id"]

In [ ]:
prepare_exchange = next(
    exchange
    for exchange in mock_backend.exchange_timeline(include_payload=True)
    if exchange["path"] == "/api/v1/agent-tools/transfers/external:prepare"
)
show_json("Agent → Backend Prepare 요청/응답", prepare_exchange)

assert prepare_exchange["request"]["from_account_id"] == SELECTED_FROM_ACCOUNT_ID
assert (
    prepare_exchange["request"]["to_recipient_candidate_id"]
    == SELECTED_RECIPIENT_CANDIDATE_ID
)
assert "to_recipient_id" not in prepare_exchange["request"]
assert prepare_exchange["request"]["amount"] == 100000

## 8. Step 8·9 실행 — 승인 후 인증 대기(4차 중단)

승인(`approved`)으로 Resume하면 Agent는 인증 Context를 생성하고 인증 요청 Webhook을 보낸 뒤 멈춘다.

In [ ]:
from agent.runtime.hitl import ExecutionResumeRequest

resume_approval_request = {
    "request_id": "req_external_resume_approval_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "resume": {
        "type": "approval",
        "confirmation_id": CONFIRMATION_ID,
        "approval_outcome": "approved",
    },
}
show_json("Backend → Agent 승인 Resume", resume_approval_request)

waiting_auth = await testbed.resume(
    THREAD_ID, ExecutionResumeRequest.model_validate(resume_approval_request)
)
show_json("4차 중단 결과(인증 대기)", waiting_auth.model_dump(mode="json"))

assert waiting_auth.status == "waiting"
assert waiting_auth.pending_interaction["type"] == "authentication"
AUTH_CONTEXT_ID = waiting_auth.pending_interaction["auth_context_id"]

## 9. Step 10·11 실행 — 인증 성공 후 실행과 결과

인증 성공(`verified`)으로 Resume하면 Agent는 송금을 실행하고 결과 Webhook을 전송한 뒤 완료한다.

In [ ]:
resume_auth_request = {
    "request_id": "req_external_resume_auth_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "resume": {
        "type": "authentication",
        "auth_context_id": AUTH_CONTEXT_ID,
        "auth_status": "verified",
    },
}
show_json("Backend → Agent 인증 Resume", resume_auth_request)

completed = await testbed.resume(
    THREAD_ID, ExecutionResumeRequest.model_validate(resume_auth_request)
)
show_json("최종 결과", completed.model_dump(mode="json"))

assert completed.status == "completed"

In [ ]:
execute_exchange = next(
    exchange
    for exchange in mock_backend.exchange_timeline(include_payload=True)
    if exchange["path"] == "/api/v1/agent-tools/transfers/external"
    and exchange["method"] == "POST"
)
show_json("Agent → Backend 실행 요청/응답", execute_exchange)

result_webhook = mock_backend.exchange_timeline(include_payload=True)[-1]["request"]
show_json("Agent → Backend 결과 Webhook", result_webhook)

assert result_webhook["event_type"] == "component"
assert (
    result_webhook["metadata"]["ui"]["payload"]["transaction_id"] == "txn_testbed_002"
)

### 완료 시점의 Agent State

수취인 참조, 출금 계좌, 금액, 거래 ID가 모두 반영되고 중간에 쓰인 `input_request_id`는 제거되어야 한다.

In [ ]:
completed_state = await testbed.state(THREAD_ID)
show_json(
    "완료 시점 시스템 State",
    {
        "current_step_id": completed_state.get("current_step_id"),
        "route_key": completed_state.get("route_key"),
        "status": completed_state.get("status"),
    },
)
show_json("완료 시점 Workflow Data", completed_state["data"])
assert (
    completed_state["data"]["to_recipient_candidate_id"]
    == SELECTED_RECIPIENT_CANDIDATE_ID
)
assert completed_state["data"]["from_account_id"] == SELECTED_FROM_ACCOUNT_ID
assert completed_state["data"]["input_request_id"] is None

## 10. 전체 통신 순서와 자동 확인

마지막으로 Workflow가 실제로 호출한 모든 HTTP 요청을 순서대로 확인한다. 인증 Token과 Secret Header는 출력하지 않는다.

In [ ]:
timeline = testbed.request_timeline()
show_json("Agent HTTP 호출 순서", timeline)

paths = [item["path"] for item in timeline]
checks = {
    "수취인 확정 API 1회": paths.count("/api/v1/agent-tools/recipients:resolve") == 1,
    "출금 계좌 확인 API 1회": paths.count("/api/v1/agent-tools/accounts") == 1,
    "Prepare API 1회": paths.count("/api/v1/agent-tools/transfers/external:prepare")
    == 1,
    "Auth Context 생성 1회": paths.count("/api/v1/agent-tools/auth-contexts") == 1,
    "실행 API 1회": paths.count("/api/v1/agent-tools/transfers/external") == 1,
    "결과 component Webhook 전송": timeline[-1].get("event_type") == "component",
    "Workflow 완료": completed.status == "completed",
}
show_json("자동 확인 결과", checks)
assert all(checks.values())

## 11. 다른 Route는 무엇이 달라지는가

| 지점 | Backend 응답/사용자 선택 | 다음 동작 |
| --- | --- | --- |
| 수취인 확정 | 이름 힌트 없음 | `resolve_recipient_hint` 호출 없이 바로 선택 화면 |
| 수취인 확정 | `resolved` | 선택 UI 없이 바로 출금 계좌 확인 |
| 수취인 선택 | 기존 수취인 선택 | `to_recipient_id`만 채워짐 |
| 수취인 선택 | 신규 후보 선택 | `to_recipient_candidate_id`만 채워짐(이 Notebook 기준 Scenario) |
| 계좌 확인 | `no_accounts` | 빈 계좌 UI를 전송하고 종료 |
| Prepare | `correction_required` (대상 1개) | 선택 화면 없이 바로 해당 항목 재입력 |
| Prepare | `correction_required` (대상 2개 이상) | 정정 대상 선택 화면 표시 |
| Prepare | `blocked` | 차단 안내 후 종료 |
| 승인 | `cancelled` | 인증·실행 없이 종료 |
| 인증 | `failed`/`expired` | 재시도 선택 화면 표시 |
| 실행 | `reauthentication_required` | Prepare·승인 없이 인증만 재수행 |

이 Route들은 `agent/tests/test_external_transfer_reference_workflow.py`에서 모두 자동 검증한다(12개 Scenario). Notebook은 본인이체와 가장 다른 지점인 "수취인 선택 → 계좌 선택 → 승인 → 인증 → 실행" 경로를 기준 Scenario로 사용한다.

## 12. 실제 Backend로 전환하기

아래 Cell은 기본적으로 실행하지 않는다. 실제 Backend 테스트 Context가 준비되면 환경변수를 설정하고 `RUN_REAL_BACKEND = True`로 변경한다.

이 단계는 Agent Tool API와 Webhook까지 검증한다. Frontend 입력과 Backend 검증을 포함한 전체 E2E는 Backend 실행 시작·Resume Endpoint와 브라우저 테스트에서 확인한다.

In [ ]:
RUN_REAL_BACKEND = False

if RUN_REAL_BACKEND:
    required_variables = [
        "BACKEND_BASE_URL",
        "AGENT_SERVICE_TOKEN",
        "AGENT_WEBHOOK_SECRET",
        "TESTBED_CHAT_SESSION_ID",
        "TESTBED_EXECUTION_CONTEXT_ID",
    ]
    missing = [name for name in required_variables if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Backend Mode 환경변수가 없습니다: {missing}")

    real_config = BackendClientConfig(
        base_url=os.environ["BACKEND_BASE_URL"],
        agent_service_token=SecretStr(os.environ["AGENT_SERVICE_TOKEN"]),
        agent_webhook_secret=SecretStr(os.environ["AGENT_WEBHOOK_SECRET"]),
    )
    real_testbed = create_external_transfer_backend_testbed(
        real_config, thread_id="thread_external_real_001"
    )
    real_start = await real_testbed.start(
        message=USER_MESSAGE,
        request_id="req_external_real_001",
        chat_session_id=os.environ["TESTBED_CHAT_SESSION_ID"],
        execution_context_id=os.environ["TESTBED_EXECUTION_CONTEXT_ID"],
    )
    show_json("실제 Backend 시작 결과", real_start.model_dump(mode="json"))
    await real_testbed.aclose()
else:
    print("Mock Backend Mode — 실제 Backend 호출 없음")

## 13. Testbed 종료

In [ ]:
await testbed.aclose()
print("타인송금 단계별 Testbed 종료 완료")